In [ ]:
import os

GAMMA_HOME = "/u/mawson-r0/eric/GAMMA/GAMMA_SOFTWARE-20251203"

os.environ["GAMMA_HOME"] = GAMMA_HOME
os.environ["MSP_HOME"] = f"{GAMMA_HOME}/MSP"
os.environ["ISP_HOME"] = f"{GAMMA_HOME}/ISP"
os.environ["DIFF_HOME"] = f"{GAMMA_HOME}/DIFF"
os.environ["LAT_HOME"] = f"{GAMMA_HOME}/LAT"
os.environ["GEO_HOME"] = f"{GAMMA_HOME}/GEO"
os.environ["DISP_HOME"] = f"{GAMMA_HOME}/DISP"
os.environ["IPTA_HOME"] = f"{GAMMA_HOME}/IPTA"

gamma_bins = [
    f"{GAMMA_HOME}/MSP/bin",
    f"{GAMMA_HOME}/ISP/bin",
    f"{GAMMA_HOME}/DIFF/bin",
    f"{GAMMA_HOME}/LAT/bin",
    f"{GAMMA_HOME}/GEO/bin",
    f"{GAMMA_HOME}/DISP/bin",
    f"{GAMMA_HOME}/IPTA/bin",
]

old_path = os.environ.get("PATH", "")
old_path_clean = ":".join(
    p for p in old_path.split(":")
    if "GAMMA_SOFTWARE-" not in p
)

os.environ["PATH"] = ":".join(gamma_bins + [old_path_clean])

In [ ]:
import subprocess
subprocess.run(["which", "par_RCM_SLC"], capture_output=True, text=True).stdout

In [ ]:
ref_slc = "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/RCM/SLC/Track137_S02_dsc/20260210_035805.slc"
ref_par = "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/RCM/SLC/Track137_S02_dsc/20260210_035805.par"

sec_ref_slc = "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/RCM/SLC/Track137_S02_dsc/20260214_035742.slc"
sec_ref_par = "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/RCM/SLC/Track137_S02_dsc/20260214_035742.par"


# Jutulstramen intfereogram #1 
# ref_slc = "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/RCM/SLC/Track137_S02_dsc/20260117_035805.slc"
# ref_par = "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/RCM/SLC/Track137_S02_dsc/20260117_035805.par"

# sec_ref_slc = "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/RCM/SLC/Track137_S02_dsc/20260121_035742.slc"
# sec_ref_par = "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/RCM/SLC/Track137_S02_dsc/20260121_035742.par"


#---Shirase Track----#
# ref_slc = "/u/fram-r0/eric/ALOS-2/ALOS2_2022/SINA_TEST/test_pair/41185.slc"
# ref_par = "/u/fram-r0/eric/ALOS-2/ALOS2_2022/SINA_TEST/test_pair/41185.par"

# #sec
# sec_ref_slc = "/u/fram-r0/eric/ALOS-2/ALOS2_2022/SINA_TEST/test_pair/41392.slc"
# sec_ref_par = "/u/fram-r0/eric/ALOS-2/ALOS2_2022/SINA_TEST/test_pair/41392.par"


#test

In [ ]:
import os 
import subprocess
import yaml
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import numpy as np
import shutil
import py_gamma as pg 
#import py_gamma2019 as pg9
from IPython.display import Image,display
from PIL import Image as PILImage
from rcm_tools.create_isp_par import create_isp_par
from scipy.signal import medfilt
from astropy.convolution import convolve, Box2DKernel

# Custom utility functions
from st_release.median_filter_off import median_filter_off
from st_release.fill_nodata import fill_nodata

In [ ]:
# ---------------------------------------------------

data_dir = "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/RCM/SLC/Track137_S02_dsc/"
out_dir = "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/RCM/SLC/Track137_S02_dsc/"

# data_dir = "/u/fram-r0/eric/ALOS-2/ALOS2_2022/SINA_TEST/test_pair/"
# out_dir = "/u/fram-r0/eric/ALOS-2/ALOS2_2022/SINA_TEST/test_pair/"

ref  = os.path.splitext(os.path.basename(ref_slc))[0]
sec = os.path.splitext(os.path.basename(sec_ref_slc))[0]

# ---------------------------------------------------

slc_exists = os.path.exists(ref_slc)
sec_exists = os.path.exists(sec_ref_slc)

slc_exists, sec_exists


In [ ]:
# - Make 1000 x 1000 pixel clip of the reference SLC for visualization
pg.SLC_copy(ref_slc, ref_par,
                os.path.join(data_dir, 'clip_to_plot.slc'),
                os.path.join(data_dir, 'clip_to_plot.par'),
                '-','-',0,1000,0,1000,'-',0)

# make a plot of a different region 
#SLC_copy cuts out slc images and allows to reformat them
# pg.SLC_copy(ref_slc, ref_par,
#                 os.path.join(data_dir, 'clip_to_plot.slc'),
#                 os.path.join(data_dir, 'clip_to_plot.par'),
#                 '-','-',2000,1000,17000,1000,'-',0)

# - Calculate a multi-look intensity (MLI) image from the reference SLC
pg.multi_look(os.path.join(data_dir, 'clip_to_plot.slc'),
                os.path.join(data_dir, 'clip_to_plot.par'),
                os.path.join(data_dir, 'clip_to_plot.pwr1'),
                os.path.join(data_dir, 'clip_to_plot.pwr1.par'),
                1, 1      # number of range/azimuth looks
                )



#- Generate 8-bit greyscale raster image of intensity multi-looked SLC
pg.raspwr(os.path.join(data_dir, 'clip_to_plot.pwr1'), 1000)

output_image_path_bmp = os.path.join(data_dir, 'clip_to_plot.pwr1.bmp')
output_image_path_png = os.path.join(data_dir, 'clip_to_plot.pwr1.png')
    
# Convert BMP to PNG
with PILImage.open(output_image_path_bmp) as img:
    img.save(output_image_path_png)

# Display the image in the notebook
display(Image(filename=output_image_path_png))

In [ ]:
# Define and check path to yaml file

# ---------------------------------------------------

home_dir_path = os.path.expanduser('~/sinacode')
param_yaml = os.path.join(home_dir_path, 'rcm_insar_proc', 'intf_processing_parameters.yml')
print(param_yaml)

# ---------------------------------------------------

if os.path.exists(param_yaml):
    print("File exists")
else:
    print("File does not exist")

In [ ]:
with open(param_yaml, encoding='utf8') as file:
    proc_param = yaml.load(file, Loader=yaml.FullLoader)

# - Create New ISP Parameter file
create_isp_par(data_dir, ref, sec)

# - Registration offset fields Preliminary Offset
p_doff = proc_param['preliminary_offsets_parameters']['p_doff']
s_window = proc_param['preliminary_offsets_parameters']['search_window']
skip = proc_param['preliminary_offsets_parameters']['skip']
nr = proc_param['preliminary_offsets_parameters']['nr']
naz = proc_param['preliminary_offsets_parameters']['naz']

# - Dense Offsets Parameters
# - AMPCOR Parameters: Search Window and Skip
rwin = proc_param['ampcor_parameters']['rwin']
azwin = proc_param['ampcor_parameters']['azwin']
rstep = proc_param['ampcor_parameters']['rstep']
azstep = proc_param['ampcor_parameters']['azstep']

# - Offsets Filtering Strategy
off_filter = proc_param['ampcor_parameters']['off_filter']
off_smooth = proc_param['ampcor_parameters']['off_smooth']
off_fill = proc_param['ampcor_parameters']['off_fill']

# - Interferogram Calculation Parameter
nlks_az = proc_param['interf_param']['nlks_az']
nlks_rn = proc_param['interf_param']['nlks_rn']
intf_filter = proc_param['interf_param']['filter']

**OFFSETS**

In the context of SAR processing, the terms "dense" and "sparse" offsets refer to two different approaches for estimating the displacement or motion between two SAR images, typically used in interferometry and offset tracking techniques. These offsets can be related to various phenomena, including earth surface movements (due to earthquakes, landslides, etc.), glacier flow, or other environmental changes.

**DENSE VS SPARSE OFFSETS**

<u>**Dense Offsets**</u>

**Definition**: Dense offset tracking involves calculating the displacement between two SAR images at every possible pixel or over a very fine grid. This results in a dense field of offset vectors that cover the entire image area.

**Purpose**: It's used when a high-resolution displacement field is required, allowing for detailed analysis of surface movements.

**Technique**: Dense offset tracking usually employs cross-correlation techniques on small patches or windows across the entire image pair to estimate the local shifts that best align the two images.
*Applications**: Dense offsets are particularly useful in studying detailed deformation patterns, such as those caused by landslides, glacier flows, or detailed urban infrastructure shifts.

<u>**Sparse Offsets**</u>  

**Definition**: Sparse offset tracking, on the other hand, calculates displacements only at selected points or features within the SAR images, rather than across the entire image. These points are typically areas with high coherence or distinctive features that can be easily matched between images.

**Purpose**: It provides a way to estimate larger-scale or regional displacements where high-resolution detail is not necessary, or in areas where the signal-to-noise ratio makes dense offset tracking unreliable.

**Technique**: This method relies on identifying distinct features or coherent patches within the SAR images and tracking these features across the image pair to estimate displacement.

**Applications**: Sparse offsets are useful for monitoring large-scale earth movements, such as tectonic shifts, where the focus is on regional trends rather than detailed local movements.

**Here we use the Gamma function `offset_pwr` to generate sparse offsets, lower in the workflow we will generate dense offsets**

`pg.offset_pwr` estimates offsets on a regular grid. Offsets refer to the differences in position between two SAR images. These offsets are important for tasks such as image registration and motion compensation.

The function takes several input parameters, including the file paths for the reference and secondary SAR images, as well as their associated parameter files.

The `s_window` parameter represents the size of the window used for offset estimation. This window is applied to both the reference and secondary images. The `nr` and `naz` parameters specify the number of range and azimuth offsets to be estimated.

In [ ]:
#p_doff == False
# - Compute Dense Offsets
# - Estimate offsets on a regular grid
pg.offset_pwr(os.path.join(data_dir, f'{ref}.slc'),
                os.path.join(data_dir, f'{sec}.slc'),
                os.path.join(data_dir, f'{ref}.par'),
                os.path.join(data_dir, f'{sec}.par'),
                os.path.join(data_dir, f'{ref}-{sec}.par'),
                os.path.join(data_dir, 'sparse_offsets'),
                os.path.join(data_dir, 'sparse_offsets.ccp'),
                s_window, s_window,
                os.path.join(data_dir, 'sparse_offsets.txt'),
                '-', nr, naz
                )

`pg.offset_fit` refines initial offset measurements between two SAR images using a polynomial fitting approach. 

This refinement process involves:
- Reading the initial offsets from a file.
- Utilizing cross-correlation peaks to improve the accuracy of these offsets.
- Applying a cubic polynomial fit to produce more precise offset values.

This function improves the alignment between SAR images, which is essential for the folloiwing interferometric processing and analysis.

In [ ]:
# - Estimate range and azimuth offset polynomial
# - Update ISP parameter file - offsets polynomial
pg.offset_fit(os.path.join(data_dir, f'sparse_offsets'),
                os.path.join(data_dir, f'sparse_offsets.ccp'),
                os.path.join(data_dir, f'{ref}-{sec}.par'),
                '-', '-', '-', 3)

`SLC_interp` is used to register a Single Look Complex (SLC) image to a reference geometry. Specifically, this involves aligning SLC-2 (the secondary image) to the geometry of SLC-1 (the reference image).
Inputs:
- The secondary SLC file (`{sec}.slc`).
- The reference parameter file (`{ref}.par`).
- The secondary parameter file (`{sec}.par`).
- The output parameter file for the registered image (`{ref}-{sec}.par`).
- The registered secondary SLC file (`{sec}.reg.slc`).
- The registered secondary parameter file (`{sec}.reg.par`).

 `create_isp_par` generates a new Interferometric SAR Processor (ISP) parameter file.

 This function can take around 5 minutes for one image pair.

In [ ]:
# - SLC_interp - registers SLC-2 to the reference geometry,
# -              that is the geometry of SLC-1.
pg.SLC_interp(os.path.join(data_dir, f'{sec}.slc'),
                os.path.join(data_dir, f'{ref}.par'),
                os.path.join(data_dir, f'{sec}.par'),
                os.path.join(data_dir, f'{ref}-{sec}.par'),
                os.path.join(data_dir, f'{sec}.reg.slc'),
                os.path.join(data_dir, f'{sec}.reg.par'),
                '-', '-', 0, 7
                )
# - Create New ISP Parameter file
create_isp_par(data_dir, ref, f'{sec}.reg')

In [ ]:
import scipy
print(scipy.__version__)

`compute_dense_offsets` calculates **dense offsets** (defined above) between a reference Synthetic Aperture Radar (SAR) Single Look Complex (SLC) image and a secondary SLC image. This is used to measure the displacement between the two images in SAR processing.

This function can take around 6 minutes for one image pair.

## .reg.par file was not in the right directroy which gives you issues, it appears in the data_dir rather than the test_out dir

In [ ]:
def compute_dense_offsets(data_dir: str, out_dir: str,
                          ref: str, sec: str,
                          off_filter: int = 1,
                          rwin: int = 64, azwin: int = 64,
                          rstep: int = None, azstep: int = None,
                          off_smooth: bool = False,
                          off_fill: bool = False) -> None:
    """
    Compute Dense Offsets Map between a reference SLC and a secondary SLC
    :param data_dir: absolute path to directory containing input data
    :param out_dir: absolute path to output directory
    :param ref: reference SLC [code]
    :param sec: secondary SLC [code]
    :param off_filter: outlier identification strategy [1, 2]
    :param rwin: range window size [default: 64]
    :param azwin: azimuth window size [default: 64]
    :param rstep: range window step [skip - if not set equal to rwin/2]
    :param azstep: azimuth window step [skip - if not set equal to azwin/2]
    :param off_smooth: smooth offsets map [True, False] - def False
    :param off_fill: fill offsets map missing values [True, False] - def False
    :return: None
    """
    # - Offsets Processing Parameters
    filter_strategy = off_filter    # - Offsets Filtering Strategy
    smooth_off = off_smooth         # - Smooth Offsets Map
    fill_off = off_fill             # - Fill Offsets Map
    pair_name = f'{ref}-{sec}'      # - Pair name
    # - Range and azimuth offset polynomial estimation with offset_fit
    off_weight = 'ccp'  # - Offsets Fit weights cross-corr or SNR [ccp, snr]
    poly_order = 3      # - Polynomial order for offset estimation

    if off_filter not in [1, 2]:
        raise ValueError('Invalid Offsets Filter Strategy. Must be 1 or 2.')

    # - Set Skip Value equal to half of Search Window
    if rstep is None:
        rstep = rwin // 2

    if azstep is None:
        azstep = azwin // 2

    # - Estimates the range and azimuth registration offset fields
    # - on a preliminary coarse resolution grid
    print('#  - Estimate the range and azimuth offset fields.')
    print(f'# - Range: Search Window: {rwin}, Skip: {rstep}')
    print(f'# - Azimuth Search Window: {azwin}, Skip: {azstep}')
    pg.offset_pwr_tracking(
        os.path.join(data_dir, f'{ref}.slc'),
        os.path.join(data_dir, f'{sec}.slc'),
        os.path.join(data_dir, f'{ref}.par'),
        os.path.join(data_dir, f'{sec}.par'),
        os.path.join(data_dir, f'{pair_name}.par'),
        os.path.join(out_dir, f'{pair_name}.offmap'),
        os.path.join(out_dir, f'{pair_name}.offmap.ccp'),
        rwin, azwin,
        os.path.join(out_dir, f'{pair_name}.offmap.txt'),
        '-', '-', rstep, azstep, '-', '-', '-', '-', '-', '-',
    )

    # - Read the offset parameter file
    off_param = pg.ParFile(os.path.join(out_dir, f'{pair_name}.par')) #changed directories to data_dir from out_dir 
    rn_min = int(off_param.par_dict['offset_estimation_starting_range'][0])
    rn_max = int(off_param.par_dict['offset_estimation_ending_range'][0])
    az_min = int(off_param.par_dict['offset_estimation_starting_azimuth'][0])
    az_max = int(off_param.par_dict['offset_estimation_ending_azimuth'][0])
    rn_smp = int(off_param.par_dict['offset_estimation_range_samples'][0])
    az_smp = int(off_param.par_dict['offset_estimation_azimuth_samples'][0])
    rn_spacing = int(off_param.par_dict['offset_estimation_range_spacing'][0])
    az_spacing = int(off_param.par_dict['offset_estimation_azimuth_spacing'][0])

    # - Verify that offsets parameter file has been modified
    # - by offset_pwr_tracking
    if rn_max == 0:
        # - Read Secondary Parameter file
        sec_param = pg.ParFile(os.path.join(data_dir, f'{ref}.par'))
        rn_smp_sec = int(sec_param.par_dict['range_samples'][0])
        az_smp_sec = int(sec_param.par_dict['azimuth_lines'][0])
        rn_smp = int(rn_smp_sec / rstep)
        az_smp = int(az_smp_sec / azstep)

        if rn_smp * rstep > rn_smp_sec:
            rn_smp -= 1
        if az_smp * azstep > az_smp_sec:
            az_smp -= 1
        rn_max = rn_smp * rstep - 1
        az_max = az_smp * azstep - 1
        print(f'# - Range Sample: {rn_smp}')
        print(f'# - Azimuth Sample: {az_smp}')
        print(f'# - Ending Range: {rn_max}')
        print(f'# - Ending Azimuth: {az_max}')

        # - Update Offsets Parameter file
        off_param.set_value('offset_estimation_ending_range', rn_max)
        off_param.set_value('offset_estimation_ending_azimuth', az_max)
        off_param.set_value('offset_estimation_range_samples', rn_smp)
        off_param.set_value('offset_estimation_azimuth_samples', az_smp)
        off_param.set_value('offset_estimation_range_spacing', rstep)
        off_param.set_value('offset_estimation_azimuth_spacing', azstep)
        off_param.set_value('offset_estimation_window_width', rwin)
        off_param.set_value('offset_estimation_window_height', azwin)
        off_param.write_par(os.path.join(out_dir, f'{pair_name}.par'))

    # - Unpack Offsets Map
    o_rn, o_az, _, _, _, snr, \
        = np.loadtxt(os.path.join(out_dir, f'{pair_name}.offmap.txt'),
                     unpack=True)
    o_rn = np.array(o_rn / rn_spacing, dtype=int)
    o_az = np.array(o_az / az_spacing, dtype=int)

    # - Initialize SNR Array
    snr_array = np.zeros((az_smp, rn_smp))
    snr_array[o_az, o_rn] = snr
    snr_array.byteswap() \
        .tofile(os.path.join(out_dir, f'{pair_name}.offmap.snr'))

    # - Run Gamma offset_fit: Range and azimuth offset polynomial estimation
    # - Note: Cross-correlation coefficients of SNR values can be set as
    # -       Linear Fit weights.
    pg.offset_fit(
        os.path.join(out_dir, f'{pair_name}.offmap'),
        os.path.join(out_dir, f'{pair_name}.offmap.{off_weight}'),
        os.path.join(out_dir, f'{pair_name}.par'),
        '-', '-', '-', poly_order, 0
    )

    # - Run Gamma offset_sub: Subtraction of polynomial
    # - from range and azimuth offset estimates
    pg.offset_sub(
        os.path.join(out_dir, f'{pair_name}.offmap'),
        os.path.join(out_dir, f'{pair_name}.par'),
        os.path.join(out_dir, f'{pair_name}.offmap.res'),
    )

    # - Run GAMMA rasmph: Generate 8-bit raster graphics image of the phase
    # - and intensity of complex data - Show Interpolated Offsets Map
    pg.rasmph(os.path.join(out_dir, f'{pair_name}.offmap.res'),
               rn_smp, '-', '-', '-', '-', '-', '-', '-',
               os.path.join(out_dir, f'{pair_name}.offmap.res.bmp'))

    # - Process Offsets Map
    # - > Remove Outliers
    # - > Apply Smoothing Filter
    # - > Fill in Nodata [Optional]


    # shutil.copy(os.path.join(out_dir, f'{pair_name}.offmap.res'),
    #         os.path.join(data_dir, f'{pair_name}.offmap.res'))



    off_map \
        = pg.read_image(os.path.join(out_dir, f'{pair_name}.offmap.res'),
                        width=rn_smp, dtype='fcomplex')

    # - Remove Erroneous Offsets s by apply Median Filter.
    if filter_strategy == 1:
        # - Filtering strategy 1 - see c_off4intf.py
        med_filt_size = 9  # - Median filter Kernel Size
        med_thresh = 1  # - Median filter Threshold
    elif filter_strategy == 2:
        # - Filtering Strategy 2 - see c_off3intf.pro
        med_filt_size = 15  # - Median filter Kernel Size
        med_thresh = 0.5  # - Median filter Threshold
    else:
        raise ValueError('# - Unknown filtering strategy selected.')
    # - Compute outlier Mask

    mask = median_filter_off(off_map, size=med_filt_size, thresh=med_thresh)

    # - Set Outliers Mask borders equal to 1
    mask[:, 0:2] = 1
    mask[:, rn_smp - 2:] = 1
    mask[0:2, :] = 1
    mask[az_smp - 2:, :] = 1

    # - Apply Outliers Mask to Offsets Map
    xoff_masked = off_map.real.copy()
    yoff_masked = off_map.imag.copy()
    xoff_masked[mask] = 0
    yoff_masked[mask] = 0

    # - Run as 5x5 median filter to locate isolated offsets values - offsets
    # - surrounded by zeros and set them to zero.
    # - Find more details in step2 of off_filter.pro
    g_mask = (mask | (medfilt(xoff_masked, 5) == 0)
              | (medfilt(yoff_masked, 5) == 0))
    xoff_masked[g_mask] = np.nan
    yoff_masked[g_mask] = np.nan

    # - Smooth Offsets Map using a 3x3 Median Filter
    xoff_masked = medfilt(xoff_masked, 3)
    yoff_masked = medfilt(yoff_masked, 3)

    # - Smooth Offsets using 7x7 Boxcar Filter
    smth_kernel_size = 7
    kernel = Box2DKernel(smth_kernel_size)
    if smooth_off:
        xoff_masked \
            = convolve(xoff_masked, kernel, boundary='extend')
        yoff_masked \
            = convolve(yoff_masked, kernel, boundary='extend')

    # - Set to NaN offsets pixels that have a zero value in
    # - either of the two directions.
    ind_zero = np.where((xoff_masked == 0) | (yoff_masked == 0))
    xoff_masked[ind_zero] = np.nan
    yoff_masked[ind_zero] = np.nan

    # - Fill Offsets Map Nodata
    if fill_off:
        print('# - Filling Gaps Offsets Map by interpolation.')
        x_mask = np.ones(np.shape(xoff_masked))
        x_mask[np.where(np.isnan(xoff_masked))] = 0
        xoff_masked = fill_nodata(xoff_masked, x_mask,
                                  max_search_dist=1000, smth_iter=10)
        y_mask = np.ones(np.shape(yoff_masked))
        y_mask[np.where(np.isnan(yoff_masked))] = 0
        yoff_masked = fill_nodata(yoff_masked, y_mask,
                                  max_search_dist=1000, smth_iter=10)

    # - Set to NaN offsets pixels to Zero
    xoff_masked[np.isnan(xoff_masked)] = 0
    yoff_masked[np.isnan(yoff_masked)] = 0

    # - Save Offsets as a complex array
    off_masked = xoff_masked + 1j * yoff_masked

    off_masked.byteswap() \
        .tofile(os.path.join(out_dir, f'{pair_name}.offmap.res.filt'))

    # - Show Smoothed Offsets Map
    pg.rasmph(os.path.join(out_dir, f'{pair_name}.offmap.res.filt'), rn_smp)


def run_sub_process(cmd: list[str]) -> None:
    """
    Run a command in a subprocess
    :param cmd: command to run
    :return: None
    """
    # - Run the command
    with open(os.devnull, 'wb') as devnull:
        subprocess.check_call(cmd,
                              stdout=devnull,
                              stderr=subprocess.STDOUT)
        
# - AMPCOR version; see github for HENSLEY

# - Compute Dense Offsets Map between the reference SLC and the
# - registered secondary SLC using GAMMA AMPCOR implementation.
compute_dense_offsets(data_dir, out_dir, ref, f'{sec}.reg',
                        off_filter=off_filter,
                        rwin=rwin, azwin=azwin,
                        rstep=rstep, azstep=azstep,
                        off_smooth=off_smooth,
                        off_fill=off_fill
                        )

# - Generate a copy of the dense offsets parameter file
shutil.copy(os.path.join(data_dir, f'{ref}-{sec}.reg.par'),
            os.path.join(data_dir, f'dense_offsets.par'))

`SLC_interp_map` resamples the registered secondary SLC to match the reference SLC using a 2-D offset map computed previously. It ensures alignment between the reference and secondary images for further SAR processing, such as interferometry.

Input Variables:
`data_dir (str)`: Absolute path to the directory containing input data.
`ref (str)`: Reference SLC image code.
`sec (str)`: Secondary SLC image code.

`create_isp_par`: Generates new parameter files necessary for interferometric SAR processing.

In [ ]:
# - Resample the registered secondary SLC to the reference SLC
# - using the using a 2-D offset map computed above.
pg.SLC_interp_map(os.path.join(data_dir, f'{sec}.reg.slc'),
                    os.path.join(data_dir, f'{ref}.par'),
                    os.path.join(data_dir, f'{sec}.reg.par'),
                    os.path.join(data_dir, f'{ref}-{sec}.reg.par'),
                    os.path.join(data_dir, f'{sec}.reg2.slc'),
                    os.path.join(data_dir, f'{sec}.reg2.par'),
                    os.path.join(data_dir, f'dense_offsets.par'),
                    os.path.join(data_dir, f'{ref}-{sec}.reg.offmap.'
                                            f'res.filt'),
                    '-', '-', 0, 7
                    )

# - Generate a new parameter file ISP offset and interferogram
# - parameter files.
# - Create New ISP Parameter file
create_isp_par(data_dir, ref, f'{sec}.reg2')

`SLC_intf` computes the interferogram between a reference SLC and a resampled secondary SLC. An interferogram represents the phase difference between two SAR images, which is crucial for applications like topographic mapping and surface deformation analysis.

Input Variables
`data_dir (str)`: Absolute path to the directory containing input data.
`ref (str)`: Reference SLC image code.
`sec (str)`: Secondary SLC image code.
`nlks_rn (int)`: Number of range looks.
`nlks_az (int)`: Number of azimuth looks.

The interferogram captures the phase differences between the two SAR images, stored in the output file coco{ref}-{sec}.
Parameters and Files:

The function utilizes parameter files for both the reference and resampled secondary SLCs (`ref.par` and `sec.reg2.par`).
An additional parameter file (`ref-sec.reg2.par`) guides the interferogram computation.
The number of range and azimuth looks (`nlks_rn` and `nlks_az`) are specified to determine the level of multi-looking, which reduces noise by averaging multiple pixels.

In [ ]:
# - Compute Interferogram
pg.SLC_intf(os.path.join(data_dir, f'{ref}.slc'),
            os.path.join(data_dir, f'{sec}.reg2.slc'),
            os.path.join(data_dir, f'{ref}.par'),
            os.path.join(data_dir, f'{sec}.reg2.par'),
            os.path.join(data_dir, f'{ref}-{sec}.reg2.par'),
            os.path.join(data_dir, f'coco{ref}-{sec}'),
            nlks_rn, nlks_az      # number of range/azimuth looks
            )

`pg.base_orbit` estimates the baseline from orbit state vectors using parameter files for a reference orbit and a secondary orbit.

*Baseline* is the distance between two positions of the satellite at different times.

*Orbit State Vectors* are mathematical descriptions of a satellite's position and velocity in space at a given time. They typically include three position coordinates (e.g., x, y, z) and three velocity components (e.g., vx, vy, vz), which together describe the satellite's trajectory and motion in its orbit.

*Reference Orbit* refers to the primary or baseline orbit used as a point of comparison. It is the main orbit against which other measurements are compared.

*Secondary Orbit* refers to the orbit that is compared against the reference orbit. This could be the orbit of a second satellite or the orbit of the same satellite at a different time.

In [ ]:
# - Estimate baseline from orbit state vectors
pg.base_orbit(os.path.join(data_dir, f'{ref}.par'),
                os.path.join(data_dir, f'{sec}.reg2.par'),
                os.path.join(data_dir, f'base{ref}-{sec}.dat'),
                )

`pg.ph_slope_base` estimates and removes the flat Earth contribution from an interferogram, a process that helps correct for large-scale phase variations caused by the curvature of the Earth. This function takes several input files related to the reference and secondary orbits and the interferogram data, processes them, and outputs a corrected interferogram.

In [ ]:
# - Estimate and Remove Flat Earth Contribution from the Interferogram
pg.ph_slope_base(os.path.join(data_dir, f'coco{ref}-{sec}'),
                    os.path.join(data_dir, f'{ref}.par'),
                    os.path.join(data_dir, f'{ref}-{sec}.reg2.par'),
                    os.path.join(data_dir, f'base{ref}-{sec}.dat'),
                    os.path.join(data_dir, f'coco{ref}-{sec}.flat'),
                    )

`pg.multi_look` calculates a multi-look intensity (MLI) image from a reference Single Look Complex (SLC) image. It takes the SLC image and its parameter file as inputs, processes them to average multiple looks in the range and azimuth directions, and produces a reduced speckle MLI image along with its parameter file.


An *MLI image* is a type of radar image where multiple looks (samples) in the range (across track) and azimuth (along track) directions are averaged. This process reduces speckle noise, which is a granular noise inherent in radar images, and enhances the image quality.

An **SLC image* contains complex-valued data representing the amplitude and phase of radar signals. It is the most basic form of radar data, capturing a single look or sample per pixel.

SLC Image File (`{ref}.slc`):
The input file containing the Single Look Complex data for the reference orbit.

Reference Parameter File (`{ref}.par`):
Contains metadata and parameters for the SLC image, such as the orbit state vectors, imaging geometry, and radar system information.

Output MLI Image File (`{ref}.pwr1`):

The output file where the multi-look intensity image will be saved.
Output MLI Parameter File (`{ref}.pwr1.par`):

The output file containing metadata and parameters for the multi-look intensity image.
Number of Range/Azimuth Looks (**nlks_rn, nlks_az**):

The **number of looks** (samples) to be averaged in the range and azimuth directions, respectively. This controls the amount of speckle reduction and the resulting resolution of the MLI image.

In [ ]:
# - Calculate a multi-look intensity (MLI) image from the reference SLC
pg.multi_look(os.path.join(data_dir, f'{ref}.slc'),
                os.path.join(data_dir, f'{ref}.par'),
                os.path.join(data_dir, f'{ref}.pwr1'),
                os.path.join(data_dir, f'{ref}.pwr1.par'),
                nlks_rn, nlks_az      # number of range/azimuth looks
                )

This block extracts the shape of an interferogram from its parameter file, specifically obtaining the number of columns (width) and the number of lines (height), which represent the dimensions of the interferogram. It reads these values from the parameter file, stores them in variables, and prints the dimensions.

In [ ]:
# - Extract interferogram shape from its parameter file
igram_param_dict \
    = pg.ParFile(os.path.join(data_dir,
                                f'{ref}-{sec}.reg2.par'),).par_dict
# - read interferogram number of columns
interf_width = int(igram_param_dict['interferogram_width'][0])
interf_lines = int(igram_param_dict['interferogram_azimuth_lines'][0])
print(f'# - Interferogram Size: {interf_lines} x {interf_width}')

# Lets plot our MLI !!

In [ ]:
# - Generate 8-bit greyscale raster image of intensity multi-looked SLC
pg.raspwr(os.path.join(data_dir, f'{ref}.pwr1'), interf_width)

output_image_path_bmp = os.path.join(data_dir, f'{ref}.pwr1.bmp')
output_image_path_png = os.path.join(data_dir, f'{ref}.pwr1.png')
    
# Convert BMP to PNG
with PILImage.open(output_image_path_bmp) as img:
    img.save(output_image_path_png)

# Display the image in the notebook
display(Image(filename=output_image_path_png))

`pg.adf` performs adaptive filtering on an interferogram using its power spectral density, taking the input interferogram file and outputting a filtered interferogram and coherence map, based on the specified interferogram width. This process enhances the quality of the interferogram by reducing noise and preserving important signal features.

In [ ]:
# if intf_filter:
# - Adaptive interferogram filter using the power spectral density
pg.adf(os.path.join(data_dir, f'coco{ref}-{sec}.flat'),
        os.path.join(data_dir, f'coco{ref}-{sec}.flat.filt'),
        os.path.join(data_dir, f'coco{ref}-{sec}.flat.coh'),
        interf_width
        )

**Let's have a look at our interferogram so far!**

In [ ]:
# - Show Filtered Interferogram    # - Show Output Interferogram
pg.rasmph_pwr(
    os.path.join(data_dir, f'coco{ref}-{sec}.flat.filt'),
    os.path.join(data_dir, f'{ref}.pwr1'), interf_width
)

# (additional plotting code just for this notebook)

output_image_path_bmp = os.path.join(data_dir, f'coco{ref}-{sec}.flat.filt.bmp')
output_image_path_png = os.path.join(data_dir, f'coco{ref}-{sec}.flat.filt.png')
   
# Convert BMP to PNG
with PILImage.open(output_image_path_bmp) as img:
    img.save(output_image_path_png)

# Display the image in the notebook
display(Image(filename=output_image_path_png))

This block estimates and removes the topographic phase from the flattened interferogram using a digital elevation model (DEM). It performs geocoding and maps the DEM data to the SAR data geometry, generating several output files necessary for further processing.

*Topographic phase* is a component of the interferometric phase that arises due to the varying terrain heights in the imaged area. Removing this phase is crucial for accurately measuring ground displacement or deformation.

*Geocoding* is the process of aligning SAR data with geographic coordinates using DEM data. This ensures that the SAR image pixels correspond to their actual geographic locations on the Earth's surface.

In [ ]:
# - Estimate and Remove Topographic Phase from
# - the flattened interferogram
dem = proc_param['DEM']['dem']
dem_path = proc_param['DEM']['path'].replace(' ', '\ ')
dem_par = os.path.join(proc_param['DEM']['path'],
                        proc_param['DEM']['par']).replace(' ', '\ ')
dem_oversmp = proc_param['DEM']['oversampling']
dem = os.path.join(dem_path, dem).replace(' ', '\ ')
data_dir_gc = data_dir.replace(' ', '\ ')

pg.gc_map(os.path.join(data_dir_gc, f'{ref}.par'),
            os.path.join(data_dir_gc, f'{ref}-{sec}.reg2.par'),
            dem_par,  # - DEM/MAP parameter file
            dem,      # - DEM data file (or constant height value)\
            # - DEM segment used.
            os.path.join(data_dir_gc, 'DEM_gc_par'),
            # - DEM segment used for output products...
            os.path.join(data_dir_gc, 'DEMice_gc'),
            # - geocoding lookup table (fcomplex)
            os.path.join(data_dir_gc, 'gc_icemap'),
            dem_oversmp, dem_oversmp,
            os.path.join(data_dir_gc, 'sar_map_in_dem_geometry'),
            '-', '-', os.path.join(data_dir_gc, 'inc.geo'),
            '-', '-', '-', '-', '2', '-'
            )

Extract DEM Size from parameter file

In [ ]:
# - Extract DEM Size from parameter file
dem_par_path = os.path.join(data_dir_gc, 'DEM_gc_par')
dem_param_dict = pg.ParFile(dem_par_path).par_dict
dem_width = int(dem_param_dict['width'][0])
dem_nlines = int(dem_param_dict['nlines'][0])

print(f'# - DEM Size: {dem_nlines} x {dem_width}')

This block performs a forward geocoding transformation using the lookup table generated two steps prior. This process aligns the SAR data with geographic coordinates by transforming the data according to the lookup table. Two geocoding operations are performed: one for the height map (`hgt_icemap`) and one for the incidence angle map (`inc`).

Geocoding transforms SAR data from radar coordinates to geographic coordinates using a lookup table. This ensures that each pixel in the SAR data corresponds to the correct location on the Earth's surface.
Lookup Table (gc_icemap):

The lookup table contains the mapping information needed to convert radar coordinates to geographic coordinates. It is created based on the DEM (Digital Elevation Model) and SAR image geometry.
Steps:
Forward Geocoding for Height Map:

The `pg.geocode` function is called to perform the geocoding transformation for the height map. It is then called again to perform the geocoding transformation for the incidence angle map.

In [ ]:
# - Forward geocoding transformation using a lookup table
pg.geocode(os.path.join(data_dir_gc, 'gc_icemap'),
            os.path.join(data_dir_gc, 'DEMice_gc'), dem_width,
            os.path.join(data_dir_gc, 'hgt_icemap'),
            interf_width, interf_lines)
pg.geocode(os.path.join(data_dir_gc, 'gc_icemap'),
            os.path.join(data_dir_gc, 'inc.geo'), dem_width,
            os.path.join(data_dir_gc, 'inc'),
            interf_width, interf_lines)

`pg.gc_map_inversion` inverts a geocoding lookup table, converting geographic coordinates back to radar coordinates. It takes the original lookup table and the dimensions of the DEM and interferogram, and outputs the inverted lookup table for further geocoding transformations or corrections.

In [ ]:
# - Invert geocoding lookup table
pg.gc_map_inversion(os.path.join(data_dir_gc, 'gc_icemap'), dem_width,
                    os.path.join(data_dir_gc, 'gc_map_invert'),
                    interf_width, interf_lines)

`pg.geocode_back` performs geocoding of the reference SLC (Single Look Complex) power image using a geocoding lookup table and then visualizes the geocoded power image. This process transforms the power image from radar coordinates to geographic coordinates, aligning it with the true geographic locations on the Earth's surface.

In [ ]:
# - Geocoding of Reference SLC power using a geocoding lookup table
pg.geocode_back(os.path.join(data_dir_gc, f'{ref}.pwr1'), interf_width,
                os.path.join(data_dir_gc, 'gc_icemap'),
                os.path.join(data_dir_gc, f'{ref}.pwr1.geo'),
                dem_width, dem_nlines)
pg.raspwr(os.path.join(data_dir_gc, f'{ref}.pwr1.geo'), dem_width)

# - Show Filtered Interferogram    # - Show Output Interferogram
pg.rasmph_pwr(
    os.path.join(data_dir, f'coco{ref}-{sec}.flat.filt'),
    os.path.join(data_dir, f'{ref}.pwr1'), interf_width
)

# (additional plotting code just for this notebook)

output_image_path_bmp = os.path.join(data_dir_gc, f'{ref}.pwr1.geo.bmp')
output_image_path_png = os.path.join(data_dir_gc, f'{ref}.pwr1.geo.png')
   
# Convert BMP to PNG
with PILImage.open(output_image_path_bmp) as img:
    img.save(output_image_path_png)

# Display the image in the notebook
display(Image(filename=output_image_path_png))

`pg.phase_sim` simulates the unwrapped interferometric phase due to surface topography using the DEM height. This process removes the phase component caused by topographic variations, leaving behind the phase related to other factors such as ground deformation.

The interferometric phase is composed of several components, including contributions from the Earth's curvature, atmospheric effects, and surface topography. To accurately measure ground deformation, it's essential to remove the phase component due to topography.

In [ ]:
# - Remove Interferometric Phase component due to surface Topography.
# - Simulate unwrapped interferometric phase using DEM height.
pg.phase_sim(os.path.join(data_dir_gc, f'{ref}.par'),
                os.path.join(data_dir_gc, f'{ref}-{sec}.reg2.par'),
                os.path.join(data_dir_gc, f'base{ref}-{sec}.dat'),
                os.path.join(data_dir_gc, 'hgt_icemap'),
                os.path.join(data_dir_gc, 'sim_phase'), 1, 0, '-'
                )

The code creates a DIFF/GEO parameter file for geocoding and differential interferometry. This parameter file is used to align and compare interferometric data, facilitating accurate analysis of ground deformation and other geophysical phenomena.

In [ ]:
# - Create DIFF/GEO parameter file for geocoding and
# - differential interferometry
pg.create_diff_par(os.path.join(data_dir_gc, f'{ref}-{sec}.reg2.par'),
                    os.path.join(data_dir_gc, f'{ref}-{sec}.reg2.par'),
                    os.path.join(data_dir_gc, 'DIFF_par'), '-', 0)

`pg.sub_phase` subtracts the topographic phase from an interferogram using a simulated phase file and a DIFF/GEO parameter file. This process results in an interferogram with the topographic phase component removed, isolating the phase differences due to other factors such as ground deformation.

In [ ]:
# - Subtract topographic phase from interferogram
pg.sub_phase(os.path.join(data_dir_gc, f'coco{ref}-{sec}.flat'),
                os.path.join(data_dir_gc, 'sim_phase'),
                os.path.join(data_dir_gc, 'DIFF_par'),
                os.path.join(data_dir_gc,
                            f'coco{ref}-{sec}.flat.topo_off'), 1
                )

Show inteferogram without topographic phase 

In [ ]:
# - Show interferogram w/o topographic phase
pg.rasmph_pwr(
    os.path.join(data_dir_gc, f'coco{ref}-{sec}.flat.topo_off'),
    os.path.join(data_dir_gc, f'{ref}.pwr1'), interf_width)

output_image_path_bmp = os.path.join(data_dir_gc, f'coco{ref}-{sec}.flat.topo_off.bmp')
output_image_path_png = os.path.join(data_dir_gc, f'coco{ref}-{sec}.flat.topo_off.png')
   
# Convert BMP to PNG
with PILImage.open(output_image_path_bmp) as img:
    img.save(output_image_path_png)

# Display the image in the notebook
display(Image(filename=output_image_path_png))

`pg.geocode_back` geocodes the output interferogram, which has had its topographic phase removed, using a reference interferogram look-up table.

In [ ]:
# - Geocode Output interferogram
# - Reference Interferogram look-up table
ref_gcmap = os.path.join(data_dir_gc, 'gc_icemap')

# - geocode interferogram
pg.geocode_back(
    os.path.join(data_dir_gc, f'coco{ref}-{sec}.flat.topo_off'),
    interf_width,  ref_gcmap,
    os.path.join(data_dir_gc, f'coco{ref}-{sec}.flat.topo_off.geo'),
    dem_width, dem_nlines, '-', 1
)

# - Show Geocoded interferogram
pg.rasmph_pwr(
    os.path.join(data_dir_gc, f'coco{ref}-{sec}.flat.topo_off.geo'),
    os.path.join(data_dir_gc, f'{ref}.pwr1.geo'),  dem_width
)

output_image_path_bmp = os.path.join(data_dir_gc, f'coco{ref}-{sec}.flat.topo_off.geo.bmp')
output_image_path_png = os.path.join(data_dir_gc, f'coco{ref}-{sec}.flat.topo_off.geo.png')
   
# Convert BMP to PNG
with PILImage.open(output_image_path_bmp) as img:
    img.save(output_image_path_png)

# Display the image in the notebook
display(Image(filename=output_image_path_png))

`pg.adf` applies an adaptive filter to the interferogram, which has had its topographic phase removed, to smooth the data and reduce noise. It then visualizes the filtered interferogram.

Adaptive Interferogram Filter (ADF) uses the power spectral density of the interferogram to selectively reduce noise while preserving important signal features. This results in a cleaner and more interpretable interferogram.

(Sam and Sina : But why isn't this geocoded?)

In [ ]:
#if intf_filter:
# - Smooth the obtained interferogram with pg.adf
# - Adaptive interferogram filter using the power spectral density.
pg.adf(
    os.path.join(data_dir, f'coco{ref}-{sec}.flat.topo_off'),
    os.path.join(data_dir, f'coco{ref}-{sec}.flat.topo_off.filt'),
    os.path.join(data_dir,
                    f'coco{ref}-{sec}.flat.topo_off.filt.coh'),
    dem_width)
# - Show filtered interferogram
pg.rasmph_pwr(os.path.join(data_dir,
                            f'coco{ref}-{sec}.flat.topo_off.filt'),
                os.path.join(data_dir, f'{ref}.pwr1'), interf_width
                )

output_image_path_bmp = os.path.join(data_dir, f'coco{ref}-{sec}.flat.topo_off.filt.bmp')
output_image_path_png = os.path.join(data_dir, f'coco{ref}-{sec}.flat.topo_off.filt.png')
   
# Convert BMP to PNG
with PILImage.open(output_image_path_bmp) as img:
    img.save(output_image_path_png)

# Display the image in the notebook
display(Image(filename=output_image_path_png))

`pg.adf` applies an adaptive filter to the interferogram, which has had its topographic phase removed, to smooth the data and reduce noise. It then visualizes the filtered interferogram.

Adaptive Interferogram Filter (ADF) uses the power spectral density of the interferogram to selectively reduce noise while preserving important signal features. This results in a cleaner and more interpretable interferogram.

(Sam and Sina: Why again?)

In [ ]:
# - Smooth Geocoded Interferogram
pg.adf(os.path.join(data_dir, f'coco{ref}-{sec}.flat.topo_off.geo'),
        os.path.join(data_dir, f'coco{ref}-{sec}.flat.topo_off.geo.filt'),
        os.path.join(data_dir, f'coco{ref}-{sec}.flat.topo_off.geo.filt.coh'),
        dem_width)

### Lets plot our final inteferogram !!

In [ ]:
pg.rasmph_pwr(
    os.path.join(data_dir, f'coco{ref}-{sec}.flat.topo_off.geo.filt'),
    os.path.join(data_dir, f'{ref}.pwr1.geo'),
    dem_width
)

output_image_path_bmp = os.path.join(data_dir, f'coco{ref}-{sec}.flat.topo_off.geo.filt.bmp')
output_image_path_png = os.path.join(data_dir, f'coco{ref}-{sec}.flat.topo_off.geo.filt.png')
    
# Convert BMP to PNG
with PILImage.open(output_image_path_bmp) as img:
    img.save(output_image_path_png)

# Display the image in the notebook
display(Image(filename=output_image_path_png))

## Now lets compute a Double Difference interferogram!!

In [ ]:
import os
import py_gamma as pg


def compute_double_difference(out_dir, igram1, igram2, ref_pwr_date):

    # ----------------------------------------------------
    # Create DOUBLE_DIFFERENCE output directory
    # ----------------------------------------------------
    dd_dir = os.path.join(out_dir, "DOUBLE_DIFFERENCE")
    os.makedirs(dd_dir, exist_ok=True)

    print(f"\nCreated/using directory:\n{dd_dir}\n")

    # ----------------------------------------------------
    # Input files
    # ----------------------------------------------------
    dem_par = os.path.join(out_dir, "DEM_gc_par")

    intf1 = os.path.join(
        out_dir,
        f"coco{igram1}.flat.topo_off.geo"
    )

    intf2 = os.path.join(
        out_dir,
        f"coco{igram2}.flat.topo_off.geo"
    )

    base1 = os.path.join(
        out_dir,
        f"base{igram1}.dat"
    )

    base2 = os.path.join(
        out_dir,
        f"base{igram2}.dat"
    )

    ref_pwr = os.path.join(
        out_dir,
        f"{ref_pwr_date}.pwr1.geo"
    )

    # ----------------------------------------------------
    # Check required files
    # ----------------------------------------------------
    required_files = [
        dem_par,
        intf1,
        intf2,
        base1,
        base2,
        ref_pwr
    ]

    for f in required_files:
        if not os.path.exists(f):
            raise FileNotFoundError(f"\nMissing file:\n{f}\n")

    # ----------------------------------------------------
    # Read width from DEM_gc_par
    # ----------------------------------------------------
    dem = pg.ParFile(dem_par)

    width = int(float(dem.par_dict["width"][0]))
    nlines = int(float(dem.par_dict["nlines"][0]))
    print(nlines)
    print("DEM width =", width)

    # ----------------------------------------------------
    # Output names
    # ----------------------------------------------------
    dd_name = (
        f"coco{igram1}-{igram2}.flat.topo_off.geo"
    )

    dd_intf = os.path.join(dd_dir, dd_name)

    dd_base = os.path.join(
        dd_dir,
        f"base{igram1}-{igram2}.dat"
    )

    dd_filt = dd_intf + ".filt"

    dd_coh = dd_intf + ".filt.coh"

    phs_geo = os.path.join(
        dd_dir,
        f"phs_{igram1}-{igram2}.geo"
    )

    # ----------------------------------------------------
    # 1. Compute double difference interferogram
    # ----------------------------------------------------
    print("Computing double-difference interferogram...\n")

    pg.comb_interfs(
        intf1,
        intf2,
        base1,
        base2,
        1,
        -1,
        width,
        dd_intf,
        dd_base
    )

    # ----------------------------------------------------
    # 2. Create unfiltered display image
    # ----------------------------------------------------
    print("Creating unfiltered display image...\n")

    pg.rasmph_pwr(
        dd_intf,
        ref_pwr,
        width
    )

    # ----------------------------------------------------
    # 3. Adaptive filtering
    # ----------------------------------------------------
    print("Filtering interferogram...\n")

    pg.adf(
        dd_intf,
        dd_filt,
        dd_coh,
        width
    )

    # ----------------------------------------------------
    # 4. Create filtered display image
    # ----------------------------------------------------
    print("Creating filtered display image...\n")

    pg.rasmph_pwr(
        dd_filt,
        ref_pwr,
        width
    )

    # ----------------------------------------------------
    # 5. Extract phase
    # ----------------------------------------------------
    print("Extracting phase...\n")

    pg.cpx_to_real(
        dd_filt,
        phs_geo,
        width,
        4
    )

    pg.raspwr(
        phs_geo,
        width
    )

    # ----------------------------------------------------
    # 6. Export phase GeoTIFF
    # ----------------------------------------------------
    print("Exporting phase GeoTIFF...\n")

    phase_tiff = dd_filt + ".phase.tiff"

    pg.data2geotiff(
        dem_par,
        phs_geo,
        2,
        phase_tiff,
        -9999
    )

    # ----------------------------------------------------
    # 7. Export coherence GeoTIFF
    # ----------------------------------------------------
    print("Exporting coherence GeoTIFF...\n")

    coh_tiff = dd_coh + ".tiff"

    # pg.data2geotiff(
    #     dem_par,
    #     dd_coh,
    #     2,
    #     coh_tiff,
    #     -9999
    # )
    # ----------------------------------------------------
# Export raw phase GeoTIFF
# ----------------------------------------------------

    # Read phase
    nlines = int(float(dem.par_dict["nlines"][0]))

    phase = np.fromfile(
        phs_geo,
        dtype=np.float32
    ).reshape(nlines, width)

    # Save temporary rainbow PNG
    phase_png = dd_filt + ".phase_rainbow.png"

    plt.imsave(
        phase_png,
        phase,
        cmap="hsv",
        vmin=-np.pi,
        vmax=np.pi
    )

    # Convert PNG to georeferenced GeoTIFF
    phase_rainbow_tiff = dd_filt + ".phase_rainbow.tiff"

    pg.data2geotiff(
        dem_par,
        phase_png,
        0,
        phase_rainbow_tiff,
        0
    )

    print("Created rainbow GeoTIFF:")
    print(phase_rainbow_tiff)

    print("Created rainbow phase GeoTIFF:")
    print(phase_rainbow_tiff)

    print("\nFinished!\n")
    print("Double-difference outputs located in:")
    print(dd_dir)


# ==========================================================
# YOUR CURRENT DIRECTORY
# ==========================================================

out_dir = (
    "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/"
    "RCM/SLC/Track137_S02_dsc"
)

# ==========================================================
# YOUR TWO INTERFEROGRAMS
# ==========================================================

igram1 = "20260117_035805-20260121_035742"

igram2 = "20260210_035805-20260214_035742"

# ==========================================================
# Reference power image
# ==========================================================

ref_pwr_date = "20260117_035805"

# ==========================================================
# RUN
# ==========================================================

compute_double_difference(
    out_dir,
    igram1,
    igram2,
    ref_pwr_date
)

## lets have a look at our double difference !!

In [ ]:
# ----------------------------------------------------
# Path to DOUBLE_DIFFERENCE directory
# ----------------------------------------------------
dd_dir = (
    "/u/fram-r0/eric/PYTHON_WORKFLOWS_EXAMPLE_RAW_DATA/"
    "RCM/SLC/Track137_S02_dsc/DOUBLE_DIFFERENCE"
)

# ----------------------------------------------------
# Double-difference interferogram name
# ----------------------------------------------------
igram1 = "20260117_035805-20260121_035742"

igram2 = "20260210_035805-20260214_035742"

# ----------------------------------------------------
# Filtered BMP created by pg.rasmph_pwr
# ----------------------------------------------------
bmp_file = os.path.join(
    dd_dir,
    f"coco{igram1}-{igram2}.flat.topo_off.geo.filt.bmp"
)

# ----------------------------------------------------
# PNG output
# ----------------------------------------------------
png_file = bmp_file.replace(".bmp", ".png")

# ----------------------------------------------------
# Convert BMP -> PNG
# ----------------------------------------------------
with PILImage.open(bmp_file) as img:
    img.save(png_file)

# ----------------------------------------------------
# Display PNG in notebook
# ----------------------------------------------------
display(Image(filename=png_file))